In [0]:
# Notebook : landing_to_bronze_hr
# Purpose: Ingest Parquet files from landing to bronze layer (Appned Only)
# Author: Virendra Dilip Tambavekar
# HRM ID : 6217
# Date : 14/05/2026

In [0]:
#Importing the required libraries
import logging
from pyspark.sql.functions import *

#Initialize Logger
logger = logging.getLogger("LandingToBronzeIngestion")
logger.setLevel(logging.INFO)

In [0]:
dbutils.widgets.text("catalog","charles_schwab_retailbrokerage_dev_team_lemma")
dbutils.widgets.text("batch_id","1")

catalog = dbutils.widgets.get("catalog")
batch_id = dbutils.widgets.get("batch_id")

#Paths
source_landing_path = f"/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg/Batch{batch_id}/hr/"
target_bronze_table = f"charles_schwab_retailbrokerage_dev_team_lemma.bronze.hr"

In [0]:
# Idempotency logic

def idempotency_cleanup_bronze(table_name, batch):
    #Remove existing records for the given batch_id to prevent duplicates on re-run.
    if spark.catalog.tableExists(table_name):
        existing_batch_count = spark.table(table_name).filter(col("_batch_id") == batch).count()
        if existing_batch_count > 0:
            logger.info(f"Idempotency: Found {existing_batch_count} existing rows for batch_id={batch} in '{table_name}'. Removing before re-append.")
            spark.sql(f"DELETE FROM {table_name} WHERE _batch_id = '{batch}'")
            logger.info(f"Idempotency: Cleanup complete. Proceeding with fresh append.")
        else:
            logger.info(f"Idempotency: No existing rows for batch_id={batch}. Safe to append.")
    else:
        logger.info(f"Idempotency: Table '{table_name}' does not exist yet. Will be created on first write.")

idempotency_cleanup_bronze(target_bronze_table, batch_id)

In [0]:
def load_landing_to_bronze() :
    logger.info(f"Landing to Bronze Ingestion for HR in Batch{batch_id}")

    try:
        #Read file
        landing_df = spark.read.parquet(source_landing_path)
        #Add Metadata Columns (_run_id carry forwarded)
        bronze_df = (
            landing_df
            .withColumn("_ingest_ts", current_timestamp())
            .drop("_landing_ts")
        )
        #Write Bronze table (Append only)
        (
            bronze_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_bronze_table)
        )
        logger.info(f"Successfully appended HR Data to {target_bronze_table}")
        return True
    except Exception as e:
        if "Path does not exist" in str(e):
            logger.info(f"Landing file not found at {source_landing_path}")
            return False
        else :
            logger.error(f"Failed to ingest the data: {str(e)}")
            raise e

file_proceed = load_landing_to_bronze()

In [0]:
if file_proceed :
    try:
        #Read data to validate
        bronze_df = spark.table(target_bronze_table)
        bronze_count = bronze_df.filter(col("_batch_id") == batch_id).count()
        total_count = bronze_df.count()

        logger.info(f"Validate successfully")
        display(bronze_df.limit(10))
        display(spark.createDataFrame([(total_count,)], ["total_count"]))
    except Exception as e:
        logger.error(f"Validation failed for HR : {str(e)}")
        raise e